# Import

In [1]:
import pandas as pd
import numpy as np
import copy

# Stock

In [6]:
# Reproducible dummy values
rng = np.random.default_rng(42)
rng

# Bestand im Jahr 2026
bestand_jahr = 2026
bestand_jahr

2026

In [7]:
segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

segmente

['MINIS',
 'KLEINWAGEN',
 'KOMPAKTKLASSE',
 'MITTELKLASSE',
 'OBERE MITTELKLASSE',
 'OBERKLASSE',
 'SUVs',
 'GELÄNDEWAGEN',
 'SPORTWAGEN',
 'MINI-VANS',
 'GROSSRAUM-VANS',
 'UTILITIES',
 'WOHNMOBILE']

In [8]:
antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]

antriebe

['Benzin',
 'Diesel',
 'Flüssiggas (LPG) (einschl. bivalent)',
 'Erdgas (CNG) (einschl. bivalent)',
 'Elektro (BEV)',
 'Hybrid',
 'Sonstige']

In [9]:
zulassungsjahre = list(range(2025, 1981, -1))  # 2025 ... 1982
zulassungsjahre

[2025,
 2024,
 2023,
 2022,
 2021,
 2020,
 2019,
 2018,
 2017,
 2016,
 2015,
 2014,
 2013,
 2012,
 2011,
 2010,
 2009,
 2008,
 2007,
 2006,
 2005,
 2004,
 2003,
 2002,
 2001,
 2000,
 1999,
 1998,
 1997,
 1996,
 1995,
 1994,
 1993,
 1992,
 1991,
 1990,
 1989,
 1988,
 1987,
 1986,
 1985,
 1984,
 1983,
 1982]

In [44]:
# Optional: slightly different scale per antrieb for more realistic-looking dummy data
antrieb_faktoren = {
    "Benzin": 1.00,
    "Diesel": 0.75,
    "Flüssiggas (LPG) (einschl. bivalent)": 0.08,
    "Erdgas (CNG) (einschl. bivalent)": 0.05,
    "Elektro (BEV)": 0.35,
    "Hybrid": 0.28,
    "Sonstige": 0.03,
}

# Optional: slightly different scale per segment
segment_faktoren = {
    "MINIS": 0.7,
    "KLEINWAGEN": 1.2,
    "KOMPAKTKLASSE": 1.4,
    "MITTELKLASSE": 1.1,
    "OBERE MITTELKLASSE": 0.8,
    "OBERKLASSE": 0.4,
    "SUVs": 1.5,
    "GELÄNDEWAGEN": 0.5,
    "SPORTWAGEN": 0.25,
    "MINI-VANS": 0.35,
    "GROSSRAUM-VANS": 0.3,
    "UTILITIES": 0.9,
    "WOHNMOBILE": 0.2,
}

bestand_2026_empirisch = {}

for segment in segmente:
    bestand_2026_empirisch[segment] = {}

    for antrieb in antriebe:
        factor = segment_faktoren[segment] * antrieb_faktoren[antrieb]

        counts = []
        for jahr in zulassungsjahre:
            alter = bestand_jahr - jahr

            # Base curve: newer cars more frequent than very old ones
            base = max(50, int(12000 / (1 + alter * 0.22)))

            # Add some random variation
            value = int(base * factor * rng.uniform(0.8, 1.2))

            counts.append(value)

        df = pd.DataFrame(
            {
                "Jahr der ersten Zulassung": zulassungsjahre,
                "Anzahl Fahrzeuge": counts,
            }
        )

        bestand_2026_empirisch[segment][antrieb] = df

print(bestand_2026_empirisch["SUVs"]["Elektro (BEV)"].head())

   Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                       2025              5216
1                       2024              5070
2                       2023              3793
3                       2022              3157
4                       2021              3215


In [45]:
bestand_2026_empirisch["GELÄNDEWAGEN"]["Benzin"]

,Jahr der ersten Zulassung,Anzahl Fahrzeuge
0,2025,4059
1,2024,4757
2,2023,4073
3,2022,3365
4,2021,3095
5,2020,2837
6,2019,2315
7,2018,2105
8,2017,1747
9,2016,2185


Wie viele Geländewagen mit Benzinantrieb gibt es aktuell?

In [46]:
bestand_2026_empirisch["GELÄNDEWAGEN"]["Benzin"]["Anzahl Fahrzeuge"].sum()

np.int64(62973)

# New vehicles

In [20]:
# Reproducible dummy values
rng = np.random.default_rng(123)

segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]

jahre_neuzugang = list(range(2026, 2051))  # 2026 ... 2050

# Dummy assumptions by segment
segment_faktoren = {
    "MINIS": 0.8,
    "KLEINWAGEN": 1.1,
    "KOMPAKTKLASSE": 1.3,
    "MITTELKLASSE": 1.0,
    "OBERE MITTELKLASSE": 0.7,
    "OBERKLASSE": 0.3,
    "SUVs": 1.4,
    "GELÄNDEWAGEN": 0.4,
    "SPORTWAGEN": 0.2,
    "MINI-VANS": 0.25,
    "GROSSRAUM-VANS": 0.2,
    "UTILITIES": 0.9,
    "WOHNMOBILE": 0.15,
}

# Dummy assumptions by powertrain
antrieb_faktoren = {
    "Benzin": 0.9,
    "Diesel": 0.5,
    "Flüssiggas (LPG) (einschl. bivalent)": 0.03,
    "Erdgas (CNG) (einschl. bivalent)": 0.02,
    "Elektro (BEV)": 1.0,
    "Hybrid": 0.7,
    "Sonstige": 0.02,
}

neuzugänge = {}

for segment in segmente:
    neuzugänge[segment] = {}

    for antrieb in antriebe:
        factor = segment_faktoren[segment] * antrieb_faktoren[antrieb]
        counts = []

        for jahr in jahre_neuzugang:
            # Simple dummy trend assumptions:
            # - Benzin / Diesel decrease over time
            # - Elektro increases
            # - Hybrid rises first, then slowly falls
            # - LPG / CNG / Sonstige remain low
            if antrieb == "Benzin":
                base = max(200, 8000 - (jahr - 2026) * 220)
            elif antrieb == "Diesel":
                base = max(50, 5000 - (jahr - 2026) * 180)
            elif antrieb == "Elektro (BEV)":
                base = 2500 + (jahr - 2026) * 260
            elif antrieb == "Hybrid":
                if jahr <= 2035:
                    base = 1800 + (jahr - 2026) * 120
                else:
                    base = 2880 - (jahr - 2035) * 90
                    base = max(300, base)
            elif antrieb == "Flüssiggas (LPG) (einschl. bivalent)":
                base = max(10, 220 - (jahr - 2026) * 8)
            elif antrieb == "Erdgas (CNG) (einschl. bivalent)":
                base = max(5, 160 - (jahr - 2026) * 7)
            else:  # Sonstige
                base = max(5, 100 - (jahr - 2026) * 2)

            value = int(base * factor * rng.uniform(0.85, 1.15))
            counts.append(max(0, value))

        df = pd.DataFrame(
            {
                "Jahr des Neuzugangs": jahre_neuzugang,
                "Anzahl Fahrzeuge": counts,
            }
        )

        neuzugänge[segment][antrieb] = df

# Example
print(neuzugänge["GELÄNDEWAGEN"]["Benzin"])

    Jahr des Neuzugangs  Anzahl Fahrzeuge
0                  2026              2680
1                  2027              2426
2                  2028              3110
3                  2029              2488
4                  2030              2227
5                  2031              2760
6                  2032              2438
7                  2033              2374
8                  2034              2167
9                  2035              2271
10                 2036              1964
11                 2037              2064
12                 2038              1783
13                 2039              1808
14                 2040              1632
15                 2041              1905
16                 2042              1606
17                 2043              1578
18                 2044              1499
19                 2045              1532
20                 2046              1296
21                 2047              1383
22                 2048           

In [21]:
neuzugänge["GROSSRAUM-VANS"]["Elektro (BEV)"]

,Jahr des Neuzugangs,Anzahl Fahrzeuge
0,2026,528
1,2027,570
2,2028,617
3,2029,593
4,2030,690
5,2031,828
6,2032,753
7,2033,755
8,2034,966
9,2035,1041


# Außerbetriebsetzungen

In [ ]:
# Reproducible dummy values
rng = np.random.default_rng(456)

jahre_ausserbetriebsetzung = list(range(2026, 2051))  # 2026 ... 2050
zulassungsjahre = list(range(2025, 1981, -1))         # 2025 ... 1982

segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]

# Dummy scaling by segment
segment_faktoren = {
    "MINIS": 0.8,
    "KLEINWAGEN": 1.2,
    "KOMPAKTKLASSE": 1.3,
    "MITTELKLASSE": 1.0,
    "OBERE MITTELKLASSE": 0.7,
    "OBERKLASSE": 0.4,
    "SUVs": 1.1,
    "GELÄNDEWAGEN": 0.5,
    "SPORTWAGEN": 0.2,
    "MINI-VANS": 0.35,
    "GROSSRAUM-VANS": 0.3,
    "UTILITIES": 0.9,
    "WOHNMOBILE": 0.15,
}

# Dummy scaling by powertrain
antrieb_faktoren = {
    "Benzin": 1.0,
    "Diesel": 0.8,
    "Flüssiggas (LPG) (einschl. bivalent)": 0.06,
    "Erdgas (CNG) (einschl. bivalent)": 0.04,
    "Elektro (BEV)": 0.2,
    "Hybrid": 0.15,
    "Sonstige": 0.03,
}

ausserbetriebsetzungen = {}

for exit_jahr in jahre_ausserbetriebsetzung:
    ausserbetriebsetzungen[exit_jahr] = {}

    for segment in segmente:
        ausserbetriebsetzungen[exit_jahr][segment] = {}

        for antrieb in antriebe:
            factor = segment_faktoren[segment] * antrieb_faktoren[antrieb]
            counts = []

            for zul_jahr in zulassungsjahre:
                alter = exit_jahr - zul_jahr

                # Dummy assumption:
                # older vehicles are more likely to be taken out of service
                # future first registrations relative to exit year are impossible
                if alter < 0:
                    base = 0
                elif alter <= 2:
                    base = 40
                elif alter <= 5:
                    base = 120
                elif alter <= 8:
                    base = 350
                elif alter <= 12:
                    base = 900
                elif alter <= 16:
                    base = 1800
                elif alter <= 20:
                    base = 2600
                elif alter <= 25:
                    base = 3200
                elif alter <= 30:
                    base = 3800
                else:
                    base = 4200

                value = int(base * factor * rng.uniform(0.85, 1.15))
                counts.append(max(0, value))

            df = pd.DataFrame(
                {
                    "Jahr der ersten Zulassung": zulassungsjahre,
                    "Anzahl Fahrzeuge": counts,
                }
            )

            ausserbetriebsetzungen[exit_jahr][segment][antrieb] = df

# Example access
print(ausserbetriebsetzungen[2026]["GELÄNDEWAGEN"]["Benzin"])
print(ausserbetriebsetzungen[2035]["SUVs"]["Diesel"])

    Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                        2025                20
1                        2024                17
2                        2023                61
3                        2022                54
4                        2021                52
5                        2020               170
6                        2019               171
7                        2018               169
8                        2017               425
9                        2016               429
10                       2015               436
11                       2014               414
12                       2013              1003
13                       2012               768
14                       2011               976
15                       2010               802
16                       2009              1172
17                       2008              1294
18                       2007              1282
19                       2006           

## Neue Variante (Exits nicht größer als der Bestand)

In [60]:
# Reproducible dummy values
rng = np.random.default_rng(456)

jahre_ausserbetriebsetzung = list(range(2026, 2051))  # 2026 ... 2050
zulassungsjahre = list(range(2025, 1981, -1))         # 2025 ... 1982

segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]

# Dummy scaling by segment
segment_faktoren = {
    "MINIS": 0.8,
    "KLEINWAGEN": 1.2,
    "KOMPAKTKLASSE": 1.3,
    "MITTELKLASSE": 1.0,
    "OBERE MITTELKLASSE": 0.7,
    "OBERKLASSE": 0.4,
    "SUVs": 1.1,
    "GELÄNDEWAGEN": 0.5,
    "SPORTWAGEN": 0.2,
    "MINI-VANS": 0.35,
    "GROSSRAUM-VANS": 0.3,
    "UTILITIES": 0.9,
    "WOHNMOBILE": 0.15,
}

# Dummy scaling by powertrain
antrieb_faktoren = {
    "Benzin": 1.0,
    "Diesel": 0.8,
    "Flüssiggas (LPG) (einschl. bivalent)": 0.06,
    "Erdgas (CNG) (einschl. bivalent)": 0.04,
    "Elektro (BEV)": 0.2,
    "Hybrid": 0.15,
    "Sonstige": 0.03,
}

ausserbetriebsetzungen = {}

for exit_jahr in jahre_ausserbetriebsetzung:
    ausserbetriebsetzungen[exit_jahr] = {}

    for segment in segmente:
        ausserbetriebsetzungen[exit_jahr][segment] = {}

        for antrieb in antriebe:
            bestand_df = bestand_2026_empirisch[segment][antrieb].copy()

            # make sure columns are numeric
            bestand_df["Jahr der ersten Zulassung"] = pd.to_numeric(
                bestand_df["Jahr der ersten Zulassung"], errors="coerce"
            )
            bestand_df["Anzahl Fahrzeuge"] = pd.to_numeric(
                bestand_df["Anzahl Fahrzeuge"], errors="coerce"
            ).fillna(0).astype(int)

            counts = []

            for _, row in bestand_df.iterrows():
                zul_jahr = int(row["Jahr der ersten Zulassung"])
                bestand_wert = int(row["Anzahl Fahrzeuge"])
                alter = exit_jahr - zul_jahr

                # future first registrations relative to exit year are impossible
                if alter < 0:
                    exit_wert = 0
                else:
                    # Exit share between 0 and 100%
                    # older vehicles get higher shares
                    if alter <= 2:
                        quote = rng.uniform(0.00, 0.02)
                    elif alter <= 5:
                        quote = rng.uniform(0.01, 0.05)
                    elif alter <= 8:
                        quote = rng.uniform(0.03, 0.10)
                    elif alter <= 12:
                        quote = rng.uniform(0.08, 0.18)
                    elif alter <= 16:
                        quote = rng.uniform(0.12, 0.28)
                    elif alter <= 20:
                        quote = rng.uniform(0.18, 0.40)
                    elif alter <= 25:
                        quote = rng.uniform(0.25, 0.55)
                    elif alter <= 30:
                        quote = rng.uniform(0.35, 0.70)
                    else:
                        quote = rng.uniform(0.45, 0.85)

                    # Optional scaling by segment and powertrain
                    quote *= segment_faktoren[segment]
                    quote *= antrieb_faktoren[antrieb]

                    # keep quote in [0, 1]
                    quote = min(max(quote, 0), 1)

                    # exits can never exceed stock
                    exit_wert = min(bestand_wert, int(bestand_wert * quote))

                counts.append(exit_wert)

            df = pd.DataFrame(
                {
                    "Jahr der ersten Zulassung": bestand_df["Jahr der ersten Zulassung"].astype(int),
                    "Anzahl Fahrzeuge": counts,
                }
            )

            ausserbetriebsetzungen[exit_jahr][segment][antrieb] = df

# Example access
print(ausserbetriebsetzungen[2026]["GELÄNDEWAGEN"]["Benzin"])
print(ausserbetriebsetzungen[2035]["SUVs"]["Diesel"])

    Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                        2025                24
1                        2024                 3
2                        2023                69
3                        2022                31
4                        2021                19
5                        2020                83
6                        2019                69
7                        2018                60
8                        2017                97
9                        2016               125
10                       2015                86
11                       2014                86
12                       2013               184
13                       2012                82
14                       2011               166
15                       2010               110
16                       2009               111
17                       2008               182
18                       2007               154
19                       2006           

# Leeres Dictionary Bestand

In [ ]:
jahre_bestand = list(range(2026, 2051))  # 2026 ... 2050
zulassungsjahre = list(range(2025, 1981, -1))  # 2025 ... 1982

segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]

bestand = {}

for jahr in jahre_bestand:
    bestand[jahr] = {}

    for segment in segmente:
        bestand[jahr][segment] = {}

        for antrieb in antriebe:
            bestand[jahr][segment][antrieb] = pd.DataFrame(
                {
                    "Jahr der ersten Zulassung": zulassungsjahre,
                    "Anzahl Fahrzeuge": [None] * len(zulassungsjahre),
                }
            )

# Example access
print(bestand[2026]["GELÄNDEWAGEN"]["Benzin"])

    Jahr der ersten Zulassung Anzahl Fahrzeuge
0                        2025             None
1                        2024             None
2                        2023             None
3                        2022             None
4                        2021             None
5                        2020             None
6                        2019             None
7                        2018             None
8                        2017             None
9                        2016             None
10                       2015             None
11                       2014             None
12                       2013             None
13                       2012             None
14                       2011             None
15                       2010             None
16                       2009             None
17                       2008             None
18                       2007             None
19                       2006             None
20           

# Berechnung Bestand

In [61]:
# - bestand_2026[segment][antrieb] -> DataFrame with:
#       "Jahr der ersten Zulassung", "Anzahl Fahrzeuge"
# - neuzugänge[segment][antrieb] -> DataFrame with:
#       "Jahr des Neuzugangs", "Anzahl Fahrzeuge"
# - ausserbetriebsetzungen[jahr][segment][antrieb] -> DataFrame with:
#       "Jahr der ersten Zulassung", "Anzahl Fahrzeuge"

segmente = [
    "MINIS",
    "KLEINWAGEN",
    "KOMPAKTKLASSE",
    "MITTELKLASSE",
    "OBERE MITTELKLASSE",
    "OBERKLASSE",
    "SUVs",
    "GELÄNDEWAGEN",
    "SPORTWAGEN",
    "MINI-VANS",
    "GROSSRAUM-VANS",
    "UTILITIES",
    "WOHNMOBILE",
]

antriebe = [
    "Benzin",
    "Diesel",
    "Flüssiggas (LPG) (einschl. bivalent)",
    "Erdgas (CNG) (einschl. bivalent)",
    "Elektro (BEV)",
    "Hybrid",
    "Sonstige",
]


def normalize_bestand_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure consistent structure for Bestand / Außerbetriebsetzungen style DataFrames.
    """
    result = df.copy()

    expected_cols = ["Jahr der ersten Zulassung", "Anzahl Fahrzeuge"]
    for col in expected_cols:
        if col not in result.columns:
            result[col] = pd.Series(dtype="int64")

    result = result[expected_cols].copy()
    result["Jahr der ersten Zulassung"] = pd.to_numeric(
        result["Jahr der ersten Zulassung"], errors="coerce"
    )
    result["Anzahl Fahrzeuge"] = pd.to_numeric(
        result["Anzahl Fahrzeuge"], errors="coerce"
    ).fillna(0)

    result = result.dropna(subset=["Jahr der ersten Zulassung"])
    result["Jahr der ersten Zulassung"] = result["Jahr der ersten Zulassung"].astype(int)
    result["Anzahl Fahrzeuge"] = result["Anzahl Fahrzeuge"].astype(int)

    return result


def get_neuzugang_value(neuzugang_df: pd.DataFrame, jahr: int) -> int:
    """
    Extract number of new registrations for a given year from the neuzugänge DataFrame.
    """
    df = neuzugang_df.copy()

    if "Jahr des Neuzugangs" not in df.columns or "Anzahl Fahrzeuge" not in df.columns:
        return 0

    df["Jahr des Neuzugangs"] = pd.to_numeric(df["Jahr des Neuzugangs"], errors="coerce")
    df["Anzahl Fahrzeuge"] = pd.to_numeric(df["Anzahl Fahrzeuge"], errors="coerce").fillna(0)

    value = df.loc[df["Jahr des Neuzugangs"] == jahr, "Anzahl Fahrzeuge"].sum()
    return int(value)


def add_neuzugang_to_bestand(bestand_df: pd.DataFrame, zugangs_jahr: int, anzahl: int) -> pd.DataFrame:
    """
    Add new registrations of one year as a new / existing first-registration cohort.
    """
    df = normalize_bestand_df(bestand_df)

    if anzahl == 0:
        return df

    if zugangs_jahr in df["Jahr der ersten Zulassung"].values:
        df.loc[
            df["Jahr der ersten Zulassung"] == zugangs_jahr,
            "Anzahl Fahrzeuge"
        ] += int(anzahl)
    else:
        new_row = pd.DataFrame(
            {
                "Jahr der ersten Zulassung": [int(zugangs_jahr)],
                "Anzahl Fahrzeuge": [int(anzahl)],
            }
        )
        df = pd.concat([df, new_row], ignore_index=True)

    df = (
        df.groupby("Jahr der ersten Zulassung", as_index=False)["Anzahl Fahrzeuge"]
        .sum()
        .sort_values("Jahr der ersten Zulassung", ascending=False)
        .reset_index(drop=True)
    )

    return df


def subtract_ausserbetriebsetzungen(bestand_df: pd.DataFrame, exits_df: pd.DataFrame) -> pd.DataFrame:
    """
    Subtract exits by first-registration year from the Bestand.
    Negative values are clipped to 0.
    """
    stock = normalize_bestand_df(bestand_df)
    exits = normalize_bestand_df(exits_df)

    merged = stock.merge(
        exits,
        on="Jahr der ersten Zulassung",
        how="outer",
        suffixes=("_bestand", "_exit"),
    ).fillna(0)

    merged["Anzahl Fahrzeuge"] = (
        merged["Anzahl Fahrzeuge_bestand"] - merged["Anzahl Fahrzeuge_exit"]
    ).clip(lower=0).astype(int)

    result = merged[["Jahr der ersten Zulassung", "Anzahl Fahrzeuge"]].copy()
    result = (
        result.groupby("Jahr der ersten Zulassung", as_index=False)["Anzahl Fahrzeuge"]
        .sum()
        .sort_values("Jahr der ersten Zulassung", ascending=False)
        .reset_index(drop=True)
    )

    return result


# ------------------------------------------------------------
# Build full Bestand dictionary
# ------------------------------------------------------------

bestand = {}

# Start year: use the provided initial stock
bestand[2026] = {}

for segment in segmente:
    bestand[2026][segment] = {}
    for antrieb in antriebe:
        bestand[2026][segment][antrieb] = normalize_bestand_df(
            bestand_2026[segment][antrieb]
        )

# Forward calculation:
# bestand[2027] = bestand[2026] + neuzugänge 2026 - ausserbetriebsetzungen 2026
# ...
# bestand[2050] = bestand[2049] + neuzugänge 2049 - ausserbetriebsetzungen 2049
for jahr in range(2027, 2051):
    vorjahr = jahr - 1
    bestand[jahr] = {}

    for segment in segmente:
        bestand[jahr][segment] = {}

        for antrieb in antriebe:
            prev_stock_df = bestand[vorjahr][segment][antrieb].copy()

            # 1) add new registrations of previous year as new cohort
            neuzugang_anzahl = get_neuzugang_value(
                neuzugänge[segment][antrieb],
                vorjahr
            )
            stock_plus_new = add_neuzugang_to_bestand(
                prev_stock_df,
                zugangs_jahr=vorjahr,
                anzahl=neuzugang_anzahl
            )

            # 2) subtract exits of previous year
            exits_df = ausserbetriebsetzungen[vorjahr][segment][antrieb]
            new_stock_df = subtract_ausserbetriebsetzungen(stock_plus_new, exits_df)

            bestand[jahr][segment][antrieb] = new_stock_df


# ------------------------------------------------------------
# Examples
# ------------------------------------------------------------

print(bestand[2026]["GELÄNDEWAGEN"]["Benzin"].head())
print(bestand[2027]["GELÄNDEWAGEN"]["Benzin"].head())
print(bestand[2035]["SUVs"]["Elektro (BEV)"].head())

   Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                       2025              4126
1                       2024              3793
2                       2023              3489
3                       2022              2606
4                       2021              2797
   Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                       2026              2680
1                       2025              4102
2                       2024              3790
3                       2023              3420
4                       2022              2575
   Jahr der ersten Zulassung  Anzahl Fahrzeuge
0                       2034              5628
1                       2033              5301
2                       2032              5863
3                       2031              5772
4                       2030              5266


In [62]:
print(
    neuzugänge["GELÄNDEWAGEN"]["Benzin"][
        neuzugänge["GELÄNDEWAGEN"]["Benzin"]["Jahr des Neuzugangs"] == 2026
    ]
)

   Jahr des Neuzugangs  Anzahl Fahrzeuge
0                 2026              2680


In [63]:
print(
    ausserbetriebsetzungen[2026]["GELÄNDEWAGEN"]["Benzin"]["Anzahl Fahrzeuge"].sum()
)


7271


In [64]:
print(bestand_2026_empirisch["GELÄNDEWAGEN"]["Benzin"]["Anzahl Fahrzeuge"].sum() )

62973


In [65]:
62973+2680-7271

58382

In [66]:
print(bestand[2027]["GELÄNDEWAGEN"]["Benzin"]["Anzahl Fahrzeuge"].sum())

55258


In [59]:
bestand[2027]["GELÄNDEWAGEN"]["Benzin"]

,Jahr der ersten Zulassung,Anzahl Fahrzeuge
0,2026,2680
1,2025,4106
2,2024,3776
3,2023,3428
4,2022,2552
5,2021,2745
6,2020,2173
7,2019,2089
8,2018,1939
9,2017,1893
